# H3. CCTV Action Recognition — 안전위반 탐지

### 개요

CCTV 영상에서 **안전위반 행동**을 자동 탐지하는 이진 분류 파이프라인을 구축합니다.

**데이터셋**: CCTV Action Recognition Dataset (Kaggle, Jonathan Nield)
- **13개 액션**: fall, grab, gun, hit, kick, lying_down, run, sit, sneak, stand, struggle, throw, walk
- **소스**: UCF Crime, NTU, YouTube CCTV 영상
- **Split 파일 포맷**: `filename.mp4 label (1 or 2)`

**안전위반 분류 기준**:
- <span style="color:#E74C3C; font-weight:bold">위반(VIOLATION) — 8종류</span>: fall, grab, gun, hit, kick, sneak, struggle, throw
- <span style="color:#27AE60; font-weight:bold">정상(NORMAL) — 5종류</span>: walk, stand, sit, run, lying_down

**파이프라인**:
1. Split 파일 파싱 & 데이터셋 구성
2. 영상 → 프레임 추출 (OpenCV 균등 샘플링)
3. 프레임 전처리 (리사이즈, 정규화)
4. <span style="color:#E74C3C; font-weight:bold">CNN 특징 추출 (MobileNetV2) → LSTM 시계열 분류</span>
5. 안전위반 탐지 & 시각화

```{admonition} 참고 문헌
:class: tip

1. "Action Recognition for Real-Time Safety Monitoring" (Kaggle Competition) — 실시간 CCTV 안전위반 탐지
2. "train-movinet-a1-on-cctv-dataset" (caturbudisantoso) — MoViNet-A1 영상 분류 모델 학습
3. Jonathan Ledur-Nield, "Human pose estimation and skeletal action recognition in CCTV" — CNN+LSTM 아키텍처
```

In [1]:
import os
import sys
import glob
import logging
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (LSTM, Dense, Dropout, BatchNormalization,
                                     Input, TimeDistributed, GlobalAveragePooling2D)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.utils import to_categorical

I0000 00:00:1780380199.160098 2507478 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780380199.221764 2507478 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1780380200.656056 2507478 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("/home/growingwithai/dev/205-research-etri/lecture/h3_debug.log", mode="w"),
    ]
)
log = logging.getLogger("CCTV-Safety")

### 실험 설정

영상 기반 액션 인식에서 <span style="color:#2980B9">프레임 해상도</span>와 <span style="color:#2980B9">시퀀스 길이</span>는 연산량과 표현력의 핵심 트레이드오프입니다.

- **IMG_SIZE=112**: 프레임을 112×112로 리사이즈합니다. 224×224 대비 픽셀 수가 1/4이므로 연산량이 크게 줄어들고, 64×64보다는 객체 형태를 충분히 식별할 수 있어 **연산량과 정보량의 균형점**입니다.
- **SEQ_LENGTH=16**: 16프레임을 하나의 시퀀스로 처리합니다. 30fps 기준 약 0.5초에 해당하며, 대부분의 액션(fall, kick 등)이 이 구간 내에서 식별 가능합니다.
- **SPLIT_RATIO="75%"**: Kaggle 데이터셋에서 제공하는 split 파일 중 75% train 분할을 사용합니다.

```{admonition} 핵심 개념
:class: tip

**안전위반 매핑** — 13개 액션을 이진 분류로 변환합니다.
- 위반(8종류) → label=1: 폭행·무기·침입 관련 액션
- 정상(5종류) → label=0: 일상적 이동·자세 액션

위반 클래스가 정상 클래스보다 많으므로 **클래스 불균형**이 발생하며, 이는后续 가중치 계산으로 보정합니다.
```

In [3]:
SPLIT_DIR   = "/home/growingwithai/dev/205-research-etri/lecture/h3_data_copy/Test_Train_Splits"
VIDEO_DIR   = "/home/growingwithai/dev/205-research-etri/lecture/h3_data_copy/Videos"
SPLIT_ID    = 1
SPLIT_RATIO = "75%"

IMG_SIZE    = 112
SEQ_LENGTH  = 16
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 1e-4
RANDOM_STATE = 42

In [4]:
SAFETY_VIOLATIONS = {"fall", "grab", "gun", "hit", "kick",
                     "sneak", "struggle", "throw"}
NORMAL_ACTIONS    = {"walk", "stand", "sit", "run", "lying_down"}
ALL_ACTIONS = sorted(SAFETY_VIOLATIONS | NORMAL_ACTIONS)

VIOLATION_LABEL = 1
NORMAL_LABEL    = 0

ACTION_TO_SAFETY = {}
for a in SAFETY_VIOLATIONS:
    ACTION_TO_SAFETY[a] = VIOLATION_LABEL
for a in NORMAL_ACTIONS:
    ACTION_TO_SAFETY[a] = NORMAL_LABEL

In [5]:
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

In [6]:
class Timer:
    def __init__(self, name):
        self.name = name
        self.t0 = time.time()
        log.info(f"[START] {self.name}")

    def __enter__(self):
        return self

    def __exit__(self, *args):
        log.info(f"[DONE] {self.name} — {time.time() - self.t0:.2f}s")

### Split 파일 파싱

데이터셋은 액션별로 별도의 split 파일을 제공합니다. 각 파일은 `filename.mp4 label` 형식이며,
label 1은 train, 2는 test를 의미합니다. 여기서는 **모든 split 파일을 순회**하여 하나의 DataFrame으로 통합하고,
<span style="color:#27AE60">액션 이름 → 안전위반 라벨 매핑</span>을 적용합니다.

In [7]:
with Timer("Split 파일 파싱"):
    records = []
    for action in ALL_ACTIONS:
        fpath = os.path.join(SPLIT_DIR, f"{action}_test_split{SPLIT_ID}.txt")
        if not os.path.exists(fpath):
            log.warning(f"  파일 없음: {fpath}")
            continue
        with open(fpath) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    fname = parts[0]
                    label = int(parts[1])
                    records.append({
                        "filename": fname,
                        "action": action,
                        "source_label": label,
                        "safety": ACTION_TO_SAFETY[action],
                        "safety_str": "VIOLATION" if ACTION_TO_SAFETY[action] == 1 else "NORMAL",
                        "video_path": os.path.join(VIDEO_DIR, fname),
                    })

    df = pd.DataFrame(records)
    log.info(f"  전체 클립: {len(df)}")
    if len(df) > 0:
        log.info(f"  액션 분포:\n{df['action'].value_counts().to_string()}")
        log.info(f"  안전위반: {len(df[df['safety'] == 1])} (VIOLATION), "
                 f"{len(df[df['safety'] == 0])} (NORMAL)")
    else:
        log.warning("  파싱된 데이터가 없습니다!")

[15:03:20] INFO — [START] Split 파일 파싱


[15:03:20] INFO —   전체 클립: 2600


[15:03:20] INFO —   액션 분포:
action
fall          200
grab          200
gun           200
hit           200
kick          200
lying_down    200
run           200
sit           200
sneak         200
stand         200
struggle      200
throw         200
walk          200


[15:03:20] INFO —   안전위반: 1600 (VIOLATION), 1000 (NORMAL)


[15:03:20] INFO — [DONE] Split 파일 파싱 — 0.02s


In [8]:
# 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
order = df['action'].value_counts().index
sns.countplot(data=df, y="action", order=order, ax=axes[0],
              palette="viridis")
axes[0].set_title("액션별 클립 수")

safety_colors = {"VIOLATION": "#F44336", "NORMAL": "#4CAF50"}
for label, grp in df.groupby("safety_str"):
    axes[1].barh(label, len(grp), color=safety_colors[label], alpha=0.8)
axes[1].set_title("안전위반 vs 정상")
axes[1].set_xlabel("클립 수")

plt.tight_layout()
plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_data_distribution.png", dpi=150)
plt.close()
log.info("  → 저장: lecture/h3_data_distribution.png")

/tmp/ipykernel_2507478/88148103.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, y="action", order=order, ax=axes[0],
/tmp/ipykernel_2507478/88148103.py:14: UserWarning: Glyph 50529 (\N{HANGUL SYLLABLE AEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/88148103.py:14: UserWarning: Glyph 49496 (\N{HANGUL SYLLABLE SYEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/88148103.py:14: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/88148103.py:14: UserWarning: Glyph 53364 (\N{HANGUL SYLLABLE KEUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/88148103.py:14: UserWarning: Glyph 47549 (\N{HANGUL SYLLABLE RIB}) missing from font(s) DejaVu Sans.
  plt.ti

[15:03:21] INFO —   → 저장: lecture/h3_data_distribution.png


### 영상 프레임 추출 — 균등 샘플링 전략

<span style="color:#E74C3C; font-weight:bold">이 단계는 전체 파이프라인에서 가장 중요한 전처리 과정입니다.</span>

**왜 연속 프레임이 아닌 균등 샘플링을 사용할까?**
- 연속된 16프레임(30fps 기준 0.5초)은 시간 범위가 너무 좁아 액션의 전체 맥락을 놓칩니다.
- 예: 160프레임 영상에서 연속 16프레임은 전체의 10%만 보지만, <span style="color:#E74C3C; font-weight:bold">균등 샘플링은 0, 10, 20, ..., 150번 프레임을 선택</span>하여 전체 시간 범위를 커버합니다.
- `np.linspace`로 시작~끝 프레임을 `seq_length`개로 균등 분할합니다.

**짧은 영상 처리**: 전체 프레임 수가 `seq_length`보다 적으면 마지막 프레임을 반복하여 길이를 맞춥니다. Zero-padding 대신 마지막 프레임 반복을 선택한 이유는 <span style="color:#27AE60">시각적 일관성</span>을 유지하기 때문입니다.

In [9]:
def extract_frames(video_path, seq_length=SEQ_LENGTH, img_size=IMG_SIZE):
    """비디오에서 균등 간격으로 seq_length개 프레임을 추출하여 리사이즈."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < 1:
        cap.release()
        return None

    # np.linspace로 시작~끝 프레임을 seq_length개로 균등 분할
    if total_frames >= seq_length:
        indices = np.linspace(0, total_frames - 1, seq_length, dtype=int)
    else:
        # 짧은 영상은 마지막 프레임 반복 (zero-padding 대신)
        indices = list(range(total_frames))
        while len(indices) < seq_length:
            indices.append(total_frames - 1)
        indices = np.array(indices)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return np.array(frames, dtype=np.float32)

In [10]:
def extract_frames_debug(video_path, seq_length=SEQ_LENGTH, img_size=IMG_SIZE):
    """프레임 추출 + 디버그 로그."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        log.warning(f"  열기 실패: {video_path}")
        return None

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = total / fps if fps > 0 else 0
    log.info(f"  영상 정보: {os.path.basename(video_path)} | "
             f"{w}x{h} @ {fps:.1f}fps | {total}프레임 | {duration:.1f}초")
    cap.release()

    return extract_frames(video_path, seq_length, img_size)

### 데이터셋 로드 — 이중 모드 설계

이 노트북은 **두 가지 모드**로 동작합니다:
1. <span style="color:#27AE60">**실제 비디오 모드**</span>: `VIDEO_DIR`에 mp4 파일이 있으면 OpenCV로 프레임을 추출합니다.
2. <span style="color:#2980B0">**시뮬레이션 모드**</span>: 비디오 파일이 없으면 랜덤 데이터로 파이프라인 전체를 검증합니다.

시뮬레이션 모드는 강의/데모 환경에서 비디오 파일 없이도 코드 실행을 보장하기 위한 설계입니다.

**클래스 가중치 계산**:
- 공식: `weight = total / (2 * class_count)`
- 위반 클립(8종류 액션)이 정상 클립(5종류 액션)보다 많으므로, <span style="color:#E74C3C; font-weight:bold">정상 클래스에 더 높은 가중치를 부여</span>하여 소수 클래스를 무시하지 않도록 합니다.

In [11]:
with Timer("데이터셋 로드"):
    has_videos = os.path.isdir(VIDEO_DIR) and len(
        glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))) > 0

    if has_videos:
        log.info("  모드: 실제 비디오 프레임 추출")
        X_frames = []
        y_labels = []
        y_actions = []
        skipped = 0

        for _, row in df.iterrows():
            frames = extract_frames(row["video_path"])
            if frames is None:
                skipped += 1
                continue
            X_frames.append(frames)
            y_labels.append(row["safety"])
            y_actions.append(row["action"])

        X_frames = np.array(X_frames) / 255.0  # [0,1] 정규화
        y_labels = np.array(y_labels)
        y_actions = np.array(y_actions)
        log.info(f"  로드 완료: {len(X_frames)}개, 스킵: {skipped}")

    else:
        log.warning("  비디오 파일 없음 → 시뮬레이션 모드")
        log.warning(f"  비디오를 {VIDEO_DIR}/ 에 배치하면 실제 모드로 전환됩니다")

        n_samples = min(100, len(df))  # 메모리 절약을 위해 샘플 수 제한
        X_frames = np.random.rand(n_samples, SEQ_LENGTH, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
        y_labels = df["safety"].values[:n_samples]
        y_actions = df["action"].values[:n_samples]
        log.info(f"  시뮬레이션 데이터: {X_frames.shape} (샘플 제한: {n_samples})")

[15:03:21] INFO — [START] 데이터셋 로드


[15:03:21] WARNING —   비디오 파일 없음 → 시뮬레이션 모드


[15:03:21] WARNING —   비디오를 /home/growingwithai/dev/205-research-etri/lecture/h3_data_copy/Videos/ 에 배치하면 실제 모드로 전환됩니다


[15:03:22] INFO —   시뮬레이션 데이터: (100, 16, 112, 112, 3) (샘플 제한: 100)


[15:03:22] INFO — [DONE] 데이터셋 로드 — 0.66s


In [12]:
# 클래스 가중치 계산
n_violation = np.sum(y_labels == VIOLATION_LABEL)
n_normal    = np.sum(y_labels == NORMAL_LABEL)
total       = len(y_labels)
class_weight = {
    NORMAL_LABEL:    total / (2 * n_normal)    if n_normal > 0 else 1.0,
    VIOLATION_LABEL: total / (2 * n_violation) if n_violation > 0 else 1.0,
}
log.info(f"  클래스 가중치: NORMAL={class_weight[NORMAL_LABEL]:.2f}, "
         f"VIOLATION={class_weight[VIOLATION_LABEL]:.2f}")

[15:03:22] INFO —   클래스 가중치: NORMAL=1.00, VIOLATION=0.50


### 프레임 시각화

추출된 프레임 시퀀스의 샘플을 확인합니다. 각 행은 하나의 비디오 클립이며,
16개 열은 시간 순서대로 균등 샘플링된 프레임입니다.

In [13]:
with Timer("프레임 시각화"):
    n_show = min(4, len(X_frames))
    fig, axes = plt.subplots(n_show, SEQ_LENGTH, figsize=(SEQ_LENGTH * 1.5, n_show * 2))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i in range(n_show):
        for j in range(SEQ_LENGTH):
            ax = axes[i, j]
            ax.imshow(X_frames[i, j])
            ax.axis("off")
            if j == 0:
                safety_tag = "VIOLATION" if y_labels[i] == 1 else "NORMAL"
                ax.set_ylabel(f"{y_actions[i]}\n({safety_tag})", fontsize=8)

    fig.suptitle("추출된 프레임 시퀀스 샘플", fontsize=14)
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_frame_samples.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h3_frame_samples.png")

[15:03:22] INFO — [START] 프레임 시각화


/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 52628 (\N{HANGUL SYLLABLE CU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 52636 (\N{HANGUL SYLLABLE CUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 46108 (\N{HANGUL SYLLABLE DOEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 54532 (\N{HANGUL SYLLABLE PEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 47112 (\N{HANGUL SYLLABLE RE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 51076 (\N{HANGUL SYLLABLE IM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/815094703.py:17: UserWarning: Glyph 49884 (\N{HANGUL SYLLABLE SI}) missing from font(s) DejaVu Sans.
  

[15:03:24] INFO —   → 저장: lecture/h3_frame_samples.png


[15:03:24] INFO — [DONE] 프레임 시각화 — 1.94s


### 학습/검증 분할

**Stratified split**을 사용하여 학습/검증셋 모두에서 위반/정상 비율을 동일하게 유지합니다.
클래스 불균형이 있는 상황에서 무작위 분할은 소수 클래스가 한쪽에 몰릴 위험이 있습니다.

In [14]:
with Timer("데이터 분할"):
    X_train, X_val, y_train, y_val = train_test_split(
        X_frames, y_labels,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_labels
    )
    log.info(f"  학습: {X_train.shape} (VIOLATION={np.sum(y_train==1)}, NORMAL={np.sum(y_train==0)})")
    log.info(f"  검증: {X_val.shape}   (VIOLATION={np.sum(y_val==1)}, NORMAL={np.sum(y_val==0)})")

[15:03:24] INFO — [START] 데이터 분할


[15:03:24] INFO —   학습: (80, 16, 112, 112, 3) (VIOLATION=80, NORMAL=0)


[15:03:24] INFO —   검증: (20, 16, 112, 112, 3)   (VIOLATION=20, NORMAL=0)


[15:03:24] INFO — [DONE] 데이터 분할 — 0.11s


### CNN + LSTM 모델 — 2단계 아키텍처

<span style="color:#E74C3C; font-weight:bold">이 섹션은 전체 파이프라인에서 가장 핵심적인 모델 설계 부분입니다.</span>

#### 왜 CNN과 LSTM을 분리하는가?
CNN+LSTM을 end-to-end로 학습하면 각 에포크마다 CNN이 동일한 프레임을 반복 처리해야 합니다.
대신 **2단계 분리** 방식을 사용합니다:

1. **1단계 — CNN 특징 추출**: MobileNetV2 (ImageNet 사전학습)로 각 프레임에서 1280차원 특징 벡터를 **미리 추출**하여 저장합니다.
2. **2단계 — LSTM 분류**: 추출된 특징 시퀀스만 LSTM에 입력하여 학습합니다.

<span style="color:#27AE60">**장점**: 메모리 절약, 학습 속도 향상 (CNN 재계산 방지)</span> — 캐글 메달 노트북들의 표준 접근법입니다.

#### MobileNetV2 특징 추출기
- **전이학습(Transfer Learning)**: ImageNet으로 사전학습된 가중치를 그대로 사용합니다.
- <span style="color:#2980B0">`include_top=False`</span>: 분류 레이어를 제거하고 특징 맵만 출력합니다.
- <span style="color:#2980B0">`trainable=False`</span>: 백본 가중치를 동결하여 학습 시간을 단축하고 오버피팅을 방지합니다.
- **GlobalAveragePooling2D**: (7, 7, 1280) 특징 맵을 (1280,) 벡터로 압축합니다. Flatten 대비 파라미터 수가 대폭 줄어듭니다.

#### LSTM 분류기
- **LSTM(128→64)**: 128유닛이 전체 시퀀스를 읽고, 64유닛이 요약합니다.
- 프레임 특징의 **시간적 순서**가 액션을 결정합니다. 예: 앞사람을 쫓아가는 것(sneak) vs 그냥 걷는 것(walk)은 시간에 따른 특징 변화 패턴으로 구분합니다.
- <span style="color:#E74C3C; font-weight:bold">`Dense(1, sigmoid)`</span>: 0~1 사이 확률을 출력하며, 값이 클수록 위반 확률이 높습니다.

In [15]:
with Timer("모델 구축"):
    # CNN 특징 추출기 (MobileNetV2 backbone)
    cnn_base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    cnn_base.trainable = False

    # GlobalAveragePooling2D로 특징 맵을 1D 벡터로 압축
    cnn_input = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = cnn_base(cnn_input, training=False)
    x = GlobalAveragePooling2D()(x)
    cnn_feat = Model(cnn_input, x, name="cnn_feature_extractor")

    feat_dim = cnn_feat.output_shape[-1]
    log.info(f"  CNN 특징 차원: {feat_dim}")

[15:03:24] INFO — [START] 모델 구축


/tmp/ipykernel_2507478/3090782494.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  cnn_base = MobileNetV2(
E0000 00:00:1780380204.142110 2507478 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[15:03:25] INFO —   CNN 특징 차원: 1280


[15:03:25] INFO — [DONE] 모델 구축 — 1.10s


In [16]:
    # CNN 특징 미리 추출 (배치 처리로 메모리 절약)
    log.info("  CNN 특징 추출 중...")
    # 학습 데이터
    n_train = X_train.shape[0]
    X_train_flat = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 3)
    train_all_feats = cnn_feat.predict(X_train_flat, batch_size=32, verbose=0)
    X_train_feat = train_all_feats.reshape(n_train, SEQ_LENGTH, feat_dim)

    # 검증 데이터
    n_val = X_val.shape[0]
    X_val_flat = X_val.reshape(-1, IMG_SIZE, IMG_SIZE, 3)
    val_all_feats = cnn_feat.predict(X_val_flat, batch_size=32, verbose=0)
    X_val_feat = val_all_feats.reshape(n_val, SEQ_LENGTH, feat_dim)

    log.info(f"  Train 특징: {X_train_feat.shape}")
    log.info(f"  Val   특징: {X_val_feat.shape}")

[15:03:25] INFO —   CNN 특징 추출 중...


[15:03:34] INFO —   Train 특징: (80, 16, 1280)


[15:03:34] INFO —   Val   특징: (20, 16, 1280)


In [17]:
    # LSTM 분류기
    lstm_model = Sequential([
        LSTM(128, input_shape=(SEQ_LENGTH, feat_dim), return_sequences=True),
        Dropout(0.3),
        BatchNormalization(),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid")  # 이진 분류: 위반 확률 [0, 1]
    ])

    lstm_model.compile(
        optimizer=Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    log.info(f"  LSTM 파라미터 수: {lstm_model.count_params():,}")

[15:03:35] INFO —   LSTM 파라미터 수: 775,809


/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### 모델 학습

LSTM 모델을 학습합니다. <span style="color:#E74C3C; font-weight:bold">클래스 가중치</span>를 적용하여 불균형 데이터에서도 소수 클래스(정상)를 무시하지 않도록 합니다.

`DebugCallback`은 5 에포크마다 학습 지표를 로그로 출력하여 진행 상황을 모니터링합니다.

In [18]:
with Timer("LSTM 학습"):
    class DebugCallback(Callback):
        def on_epoch_end(self, epoch, logs=None):
            if (epoch + 1) % 5 == 0 or epoch == 0:
                log.info(
                    f"  Epoch {epoch+1:03d}/{EPOCHS} — "
                    f"loss={logs['loss']:.4f} acc={logs['accuracy']:.4f} "
                    f"val_loss={logs['val_loss']:.4f} val_acc={logs['val_accuracy']:.4f}"
                )

    history = lstm_model.fit(
        X_train_feat, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val_feat, y_val),
        class_weight=class_weight,
        callbacks=[DebugCallback()],
        verbose=0
    )

[15:03:35] INFO — [START] LSTM 학습


[15:03:39] INFO —   Epoch 001/30 — loss=0.2836 acc=0.7125 val_loss=0.7088 val_acc=0.2000


[15:03:39] INFO —   Epoch 005/30 — loss=0.3084 acc=0.7000 val_loss=0.6806 val_acc=0.8500


[15:03:40] INFO —   Epoch 010/30 — loss=0.2720 acc=0.7250 val_loss=0.6885 val_acc=0.5000


[15:03:42] INFO —   Epoch 015/30 — loss=0.2373 acc=0.8125 val_loss=0.7364 val_acc=0.0000


[15:03:43] INFO —   Epoch 020/30 — loss=0.2523 acc=0.7875 val_loss=0.7161 val_acc=0.0500


[15:03:44] INFO —   Epoch 025/30 — loss=0.2476 acc=0.7625 val_loss=0.6702 val_acc=0.9500


[15:03:45] INFO —   Epoch 030/30 — loss=0.2278 acc=0.8125 val_loss=0.6670 val_acc=0.7500


[15:03:45] INFO — [DONE] LSTM 학습 — 10.12s


In [19]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train")
axes[1].plot(history.history["val_accuracy"], label="Val")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_training_curves.png", dpi=150)
plt.close()
log.info("  → 저장: lecture/h3_training_curves.png")

[15:03:45] INFO —   → 저장: lecture/h3_training_curves.png


### 평가 — 혼동 행렬 및 확률 분포

모델 성능을 두 가지 관점에서 평가합니다:
1. **혼동 행렬**: NORMAL/VIOLATION 간 분류 정확도를 직관적으로 보여줍니다.
2. **위반 확률 분포**: 실제 정상/위반 샘플의 예측 확률이 얼마나 잘 분리되는지 확인합니다.
   - 두 분포가 겹칠수록 오분류가 많아지며, <span style="color:#2980B0">임계값(0.5)</span> 조정이 필요할 수 있습니다.

In [20]:
with Timer("평가"):
    y_pred_prob = lstm_model.predict(X_val_feat, verbose=0).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average="weighted")

    log.info(f"  Accuracy: {acc:.4f}")
    log.info(f"  F1 (weighted): {f1:.4f}")
    log.info(f"\n{classification_report(y_val, y_pred, target_names=['NORMAL', 'VIOLATION'])}")

[15:03:45] INFO — [START] 평가


[15:03:45] INFO —   Accuracy: 0.7500


[15:03:45] INFO —   F1 (weighted): 0.8571


[15:03:45] INFO — 
              precision    recall  f1-score   support

      NORMAL       0.00      0.00      0.00         0
   VIOLATION       1.00      0.75      0.86        20

    accuracy                           0.75        20
   macro avg       0.50      0.38      0.43        20
weighted avg       1.00      0.75      0.86        20



[15:03:45] INFO — [DONE] 평가 — 0.33s


/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/growingwithai/dev/205-research-etri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [21]:
    # 혼동 행렬 + 위반 확률 분포
    cm = confusion_matrix(y_val, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["NORMAL", "VIOLATION"],
                yticklabels=["NORMAL", "VIOLATION"])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title(f"Confusion Matrix (Acc={acc:.4f}, F1={f1:.4f})")

    normal_probs    = y_pred_prob[y_val == NORMAL_LABEL]
    violation_probs = y_pred_prob[y_val == VIOLATION_LABEL]
    axes[1].hist(normal_probs,    bins=30, alpha=0.6, label="NORMAL",    color="#4CAF50")
    axes[1].hist(violation_probs, bins=30, alpha=0.6, label="VIOLATION", color="#F44336")
    axes[1].axvline(x=0.5, color="black", linestyle="--", label="Threshold=0.5")
    axes[1].set_xlabel("위반 확률")
    axes[1].set_ylabel("빈도")
    axes[1].set_title("안전위반 확률 분포")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_evaluation.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h3_evaluation.png")

/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 50948 (\N{HANGUL SYLLABLE WI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 48152 (\N{HANGUL SYLLABLE BAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 54869 (\N{HANGUL SYLLABLE HWAG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 47456 (\N{HANGUL SYLLABLE RYUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 48712 (\N{HANGUL SYLLABLE BIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/2410329650.py:22: UserWarning: Glyph 50504 (\N{HANGUL SYLLABLE AN}) missing from font(s) DejaVu

/tmp/ipykernel_2507478/2410329650.py:23: UserWarning: Glyph 48712 (\N{HANGUL SYLLABLE BIN}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_evaluation.png", dpi=150)
/tmp/ipykernel_2507478/2410329650.py:23: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_evaluation.png", dpi=150)
/tmp/ipykernel_2507478/2410329650.py:23: UserWarning: Glyph 50504 (\N{HANGUL SYLLABLE AN}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_evaluation.png", dpi=150)
/tmp/ipykernel_2507478/2410329650.py:23: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_evaluation.png", dpi=150)
/tmp/ipykernel_2507478/2410329650.py:23: UserWarning: Glyph 50948 (\N{HANGUL SYLLABLE WI}) missing from font(s) DejaVu Sa

[15:03:46] INFO —   → 저장: lecture/h3_evaluation.png


### 실시간 안전위반 탐지 데모

<span style="color:#E74C3C; font-weight:bold">실시간 추론 파이프라인</span> — 새로운 비디오 한 개를 처리하는 전체 과정입니다:

1. **프레임 추출**: 균등 샘플링으로 16프레임 획득
2. **CNN 특징 추출**: MobileNetV2로 각 프레임의 1280차원 특징 벡터 생성
3. **LSTM 분류**: 특징 시퀀스를 입력하여 위반 확률 예측
4. **임계값 비교**: <span style="color:#2980B0">확률 ≥ 0.5</span>이면 위반으로 판정

실제 서비스 환경에서는 이 함수를 스트리밍 프레임에 대해 슬라이딩 윈도우로 반복 호출합니다.

In [22]:
with Timer("실시간 탐지 데모"):
    def detect_safety_violation(video_path, model, cnn_model,
                                seq_length=SEQ_LENGTH, img_size=IMG_SIZE,
                                threshold=0.5):
        """단일 비디오에 대해 안전위반 탐지 수행."""
        frames = extract_frames(video_path, seq_length, img_size)
        if frames is None:
            return None, 0.0, None

        frames_norm = frames / 255.0

        # CNN 특징 추출
        feats = np.zeros((1, seq_length, feat_dim), dtype=np.float32)
        for t in range(seq_length):
            feats[0, t] = cnn_model(
                frames_norm[t:t+1], training=False
            ).numpy()[0]

        # LSTM 예측
        prob = model.predict(feats, verbose=0)[0, 0]
        is_violation = prob >= threshold

        return is_violation, float(prob), frames

[15:03:46] INFO — [START] 실시간 탐지 데모


[15:03:46] INFO — [DONE] 실시간 탐지 데모 — 0.00s


In [23]:
    # 검증셋에서 위반 탐지 데모
    demo_indices = np.random.choice(len(X_val), size=min(10, len(X_val)), replace=False)

    results = []
    for idx in demo_indices:
        actual = "VIOLATION" if y_val[idx] == 1 else "NORMAL"
        prob = y_pred_prob[idx]
        pred = "VIOLATION" if y_pred[idx] == 1 else "NORMAL"
        match = "O" if y_pred[idx] == y_val[idx] else "X"
        results.append({
            "actual": actual,
            "predicted": pred,
            "violation_prob": f"{prob:.3f}",
            "match": match,
        })
        log.info(f"  [{match}] 실제={actual:10s} | 예측={pred:10s} | 위반확률={prob:.3f}")

    results_df = pd.DataFrame(results)
    log.info(f"\n{results_df.to_string(index=False)}")

[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.539


[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.526


[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.509


[15:03:46] INFO —   [X] 실제=VIOLATION  | 예측=NORMAL     | 위반확률=0.500


[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.521


...

[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.511


[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.528


[15:03:46] INFO —   [X] 실제=VIOLATION  | 예측=NORMAL     | 위반확률=0.495


[15:03:46] INFO —   [O] 실제=VIOLATION  | 예측=VIOLATION  | 위반확률=0.505


[15:03:46] INFO — 
   actual predicted violation_prob match
VIOLATION VIOLATION          0.539     O
VIOLATION VIOLATION          0.526     O
VIOLATION VIOLATION          0.509     O
VIOLATION    NORMAL          0.500     X
VIOLATION VIOLATION          0.521     O
VIOLATION VIOLATION          0.529     O
VIOLATION VIOLATION          0.511     O
VIOLATION VIOLATION          0.528     O
VIOLATION    NORMAL          0.495     X
VIOLATION VIOLATION          0.505     O


In [24]:
    # 탐지 결과 시각화
    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ["#F44336" if r["actual"] == "VIOLATION" else "#4CAF50" for r in results]
    probs  = [float(r["violation_prob"]) for r in results]
    labels = [f"{r['actual']}\n{'O' if r['match']=='O' else 'X'}" for r in results]

    bars = ax.bar(range(len(probs)), probs, color=colors, alpha=0.7, edgecolor="black")
    ax.axhline(y=0.5, color="black", linestyle="--", linewidth=2, label="Threshold=0.5")
    ax.set_xticks(range(len(probs)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("위반 확률")
    ax.set_title("안전위반 탐지 결과 (빨강=VIOLATION, 초록=NORMAL)")
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.savefig("/home/growingwithai/dev/205-research-etri/lecture/h3_detection_demo.png", dpi=150)
    plt.close()
    log.info("  → 저장: lecture/h3_detection_demo.png")

[15:03:46] INFO —   → 저장: lecture/h3_detection_demo.png


/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 50948 (\N{HANGUL SYLLABLE WI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 48152 (\N{HANGUL SYLLABLE BAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 54869 (\N{HANGUL SYLLABLE HWAG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 47456 (\N{HANGUL SYLLABLE RYUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 50504 (\N{HANGUL SYLLABLE AN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2507478/22252242.py:15: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt

### 비디오 프레임 추출 유틸리티

독립적으로 사용 가능한 프레임 추출 유틸리티입니다. 지정한 FPS로 프레임을 추출하여 이미지로 저장하며, 실시간 디버깅에 유용합니다.

In [25]:
def extract_and_save_frames(video_path, output_dir, fps=5, img_size=224):
    """비디오에서 지정 FPS로 프레임을 추출하여 이미지로 저장."""
    os.makedirs(output_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        log.error(f"비디오 열기 실패: {video_path}")
        return 0

    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    skip = max(1, int(orig_fps / fps))

    log.info(f"입력: {os.path.basename(video_path)} | {w}x{h} @ {orig_fps:.1f}fps | "
             f"총 {total}프레임")
    log.info(f"추출: {fps}fps (매 {skip}프레임마다) → 약 {total // skip}프레임")

    count = 0
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % skip == 0:
            frame = cv2.resize(frame, (img_size, img_size))
            out_path = os.path.join(output_dir, f"frame_{count:05d}.jpg")
            cv2.imwrite(out_path, frame)
            count += 1
            if count % 50 == 0:
                log.info(f"  {count}프레임 추출 완료...")
        frame_idx += 1

    cap.release()
    log.info(f"완료: 총 {count}프레임 → {output_dir}")
    return count

### 결과 요약

전체 실험 결과를 요약하여 출력합니다.

In [26]:
SAVE_DIR = "/home/growingwithai/dev/205-research-etri/lecture"
print("\n" + "=" * 60)
print("결과 요약")
print("=" * 60)
print(f"  데이터셋: CCTV Action Recognition (split {SPLIT_ID}, {SPLIT_RATIO})")
print(f"  총 클립: {len(df)} (VIOLATION: {n_violation}, NORMAL: {n_normal})")
print(f"  위반 액션: {sorted(SAFETY_VIOLATIONS)}")
print(f"  정상 액션: {sorted(NORMAL_ACTIONS)}")
print(f"  모델: MobileNetV2(CNN) + LSTM")
print(f"  프레임: {SEQ_LENGTH}프레임/시퀀스, {IMG_SIZE}x{IMG_SIZE}")
print(f"")
print(f"  Accuracy: {acc:.4f}")
print(f"  F1 Score: {f1:.4f}")
mode_str = "시뮬레이션" if not has_videos else "실제 비디오"
print(f"  모드: {mode_str}")
if not has_videos:
    print(f"\n  ※ 실제 비디오를 {VIDEO_DIR}/ 에 배치하면 실제 모드로 실행됩니다")
print(f"\n  생성된 파일:")
for f in sorted(
    x for x in os.listdir(SAVE_DIR)
    if x.startswith("h3_") and (x.endswith(".png") or x.endswith(".log"))
):
    print(f"    lecture/{f}")
print("\n디버그 로그: lecture/h3_debug.log")
print("완료.")


결과 요약
  데이터셋: CCTV Action Recognition (split 1, 75%)
  총 클립: 2600 (VIOLATION: 100, NORMAL: 0)
  위반 액션: ['fall', 'grab', 'gun', 'hit', 'kick', 'sneak', 'struggle', 'throw']
  정상 액션: ['lying_down', 'run', 'sit', 'stand', 'walk']
  모델: MobileNetV2(CNN) + LSTM
  프레임: 16프레임/시퀀스, 112x112

  Accuracy: 0.7500
  F1 Score: 0.8571
  모드: 시뮬레이션

  ※ 실제 비디오를 /home/growingwithai/dev/205-research-etri/lecture/h3_data_copy/Videos/ 에 배치하면 실제 모드로 실행됩니다

  생성된 파일:
    lecture/h3_data_distribution.png
    lecture/h3_debug.log
    lecture/h3_detection_demo.png
    lecture/h3_evaluation.png
    lecture/h3_frame_samples.png
    lecture/h3_training_curves.png

디버그 로그: lecture/h3_debug.log
완료.
